In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
from math import pi
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.qubits.fluxonium import Fluxonium
from fluxoniumcr.dressed_control_fluxonium import (
    create_driven_fluxonium,
    calculate_critical_amplitude,
    calculate_polarization_and_error,
    calculate_quasienergies_and_avgenergies_around_ac,
    calculate_avoided_crossing_gap,
)

# Configuration

In [31]:
class FluxoniumDesignParameters(NamedTuple):
    EJ: float  # 2π.GHz
    EC: float  # 2π.GHz
    EL: float  # 2π.GHz


class DrivingParameters(NamedTuple):
    frequency: float  # 2π.GHz
    amplitudes: np.ndarray  # Dimensionless drive amplitude


LOOKUP_QUASIENERGY_UNIQUE_TOLERANCE = 10e-3 * (2*pi)

REPR_DESIGN_PARAMETERS = \
    FluxoniumDesignParameters(
        EJ=4.0 * (2*pi),
        EC=1.0 * (2*pi),
        EL=1.0 * (2*pi),
    )

fluxonium = Fluxonium(
    EJ=REPR_DESIGN_PARAMETERS.EJ,
    EC=REPR_DESIGN_PARAMETERS.EC,
    EL=REPR_DESIGN_PARAMETERS.EL,
    flux=0.5,
    dim=16,
    cutoff=128,
)

In [32]:
n_op = fluxonium.get_operator('charge')
qubit_frequency = fluxonium.eigenvalues[1] - fluxonium.eigenvalues[0]

# Drive amplitude unit
Ω0 = qubit_frequency/abs(n_op[0, 1])

REPR_DRIVING_PARAMETERS = (
    DrivingParameters(
        0.800 * (2*pi),
        np.linspace(0, 1.5, 301) * Ω0,
    ),
    DrivingParameters(
        0.960 * (2*pi),
        np.linspace(0, 1.5, 301) * Ω0,
    ),
    DrivingParameters(
        2.800 * (2*pi),
        np.linspace(0, 0.5, 201) * Ω0,
    ),
)

# Calculation

In [33]:
datasets = []

for case_index, parameters in enumerate(REPR_DRIVING_PARAMETERS):
    drive_frequency = parameters.frequency
    amplitude_data = parameters.amplitudes

    floquet_basis = create_driven_fluxonium(
        fluxonium,
        drive_frequency,
        phase_gauge=True,
    )
    floquet_basis.generate_lookup(
        amplitude_data,
        deg_tol=LOOKUP_QUASIENERGY_UNIQUE_TOLERANCE,
    )

    dataset = xr.Dataset(attrs=dict(
        EJ=REPR_DESIGN_PARAMETERS.EJ,
        EC=REPR_DESIGN_PARAMETERS.EC,
        EL=REPR_DESIGN_PARAMETERS.EL,
        drive_frequency=drive_frequency
    ))

    amplitude_coord = xr.DataArray(
        amplitude_data,
        dims=['amplitude'],
        attrs=dict(plot_unit=Ω0),
    )
    label_coord = xr.DataArray(
        np.arange(fluxonium.dim),
        dims=['label'],
    )
    qubit_label_coord = xr.DataArray(
        np.arange(2),
        dims=['qubit_label'],
    )

    quasienergies_data = np.array([
        floquet_basis.quasienergies(Ω)
        for Ω in amplitude_data
    ])
    dataset['quasienergy'] = xr.DataArray(
        data=quasienergies_data,
        coords=[amplitude_coord, label_coord],
        attrs=dict(
            period=drive_frequency,
            plot_unit=2*np.pi,
        )
    )
    
    # XXX: Add correction because we are calculating in the phase gauge.
    dataset['quasienergy'] -= amplitude_coord**2/(32*REPR_DESIGN_PARAMETERS.EC)

    # XXX: Extra data for small avoided crossing case.
    ac_amplitude = None
    if case_index == 1:
        ac_amplitude = calculate_avoided_crossing_gap(
            floquet_basis,
            i=0,
            j=2,
            k=-4,
            step_size=0.1 * (2*pi),
            xtol=1e-6,
        )[0]
        dataset.attrs['ac_amplitude'] = ac_amplitude

        amplitude_around_ac_data = ac_amplitude + np.linspace(-0.36, +0.36, 100)
        amplitude_around_ac_coord = xr.DataArray(
            amplitude_around_ac_data,
            dims=['amplitude_around_ac'],
            attrs=dict(plot_unit=Ω0),
        )

        quasienergies_around_ac_data, avgenergies_around_ac_data = \
            calculate_quasienergies_and_avgenergies_around_ac(
                floquet_basis,
                amplitude_around_ac_data,
            )

        dataset['quasienergy_around_ac'] = xr.DataArray(
            data=quasienergies_around_ac_data,
            coords=[amplitude_around_ac_coord, label_coord],
            attrs=dict(
                period=drive_frequency,
                plot_unit=2*np.pi,
            )
        )
    
        # XXX: Add correction because we are calculating in the phase gauge.
        dataset['quasienergy_around_ac'] -= amplitude_around_ac_coord**2/(32*REPR_DESIGN_PARAMETERS.EC)


        dataset['avgenergy_around_ac'] = xr.DataArray(
            data=avgenergies_around_ac_data,
            coords=[amplitude_around_ac_coord, label_coord],
            attrs=dict(plot_unit=2*np.pi)
        )

    polarizations_data = []
    error_rate_data = []

    for Ω in amplitude_data:
        p0, p1, y = calculate_polarization_and_error(
            fluxonium,
            floquet_basis.query(Ω),
        )
        polarizations_data.append((p0, p1))
        error_rate_data.append(y)

    dataset['polarization'] = xr.DataArray(
        data=polarizations_data,
        coords=[
            amplitude_coord,
            qubit_label_coord,
        ],
    )
    dataset['error_rate'] = xr.DataArray(
        data=error_rate_data,
        coords=[amplitude_coord],
        attrs=dict(plot_unit=1e-6)
    )

    if ac_amplitude is None:
        optimal_amplitude, _ = calculate_critical_amplitude(
            fluxonium,
            floquet_basis,
            step_size=0.1 * (2*pi),
            xtol=1e-6,
        )
        dataset.attrs['optimal_amplitude'] = optimal_amplitude
    else:
        optimal_amplitude_first, _ = calculate_critical_amplitude(
            fluxonium,
            floquet_basis,
            step_size=0.1 * (2*pi),
            xtol=1e-6,
            x1=ac_amplitude,
            minimum_energy_gap=0,
        )
        optimal_amplitude, _ = calculate_critical_amplitude(
            fluxonium,
            floquet_basis,
            step_size=0.1 * (2*pi),
            xtol=1e-6,
            x0=ac_amplitude + 0.1,
            minimum_energy_gap=0,
        )
        dataset.attrs['optimal_amplitude_first'] = optimal_amplitude_first
        dataset.attrs['optimal_amplitude'] = optimal_amplitude

    p0_series_data, p1_series_data, _ = calculate_polarization_and_error(
        fluxonium,
        floquet_basis.query_perturbative(0.0, order=3),
    )
    order_coord = xr.DataArray(
        np.arange(4),
        dims=['order'],
    )

    dataset['p0_series'] = xr.DataArray(
        data=p0_series_data,
        coords=[order_coord],
    )
    dataset['p1_series'] = xr.DataArray(
        data=p1_series_data,
        coords=[order_coord],
    )

    datasets.append(dataset)

In [36]:
parent_save_string = ",".join([
    f"EJ={REPR_DESIGN_PARAMETERS.EJ/(2*pi)}",
    f"EC={REPR_DESIGN_PARAMETERS.EC/(2*pi)}",
    f"EL={REPR_DESIGN_PARAMETERS.EL/(2*pi)}",
])

parent_directory = DATA_DIR/"three_cases"/parent_save_string
parent_directory.mkdir(parents=True, exist_ok=True)

for dataset in datasets:
    filename_string = f"drive_frequency={dataset.drive_frequency/(2*pi)}" + ".hdf5"
    dataset.to_netcdf(parent_directory/filename_string)